# Privacy & Federated Learning

Companion notebook for the [Privacy & Federated Learning lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/15-privacy-and-federated-learning).

**The idea in one sentence.** Two ways to train on sensitive data without exposing it:
**DP-SGD** adds calibrated noise (after clipping each example's gradient) so no single
record measurably changes the model, and **federated learning** keeps data on-device,
sending only model updates to be averaged.

The core mechanics:

- **DP-SGD:** clip per-example gradients (bound each record's influence), then add Gaussian
  noise — trading **utility for privacy**.
- **FedAvg:** clients train locally and the server averages their weights, **weighted by
  data count** — the data never leaves the device.

We build both from scratch, **validate the privacy–utility trade-off and FedAvg's
weighting**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — DP-SGD: clip per-example gradients, then add noise

Clipping bounds how much any single example can move the model (its sensitivity); the calibrated
Gaussian noise then masks any individual's contribution. We fit a simple linear model both ways.

In [ ]:
# tiny regression problem: y = w*x
Xd = rng.normal(size=200); w_true = 2.0
yd = w_true * Xd + rng.normal(0, 0.1, 200)

def clip(g, C):
    norm = abs(g)
    return g * min(1.0, C / (norm + 1e-12))

def dp_sgd_step(w, Xb, yb, lr, C, sigma):
    grads = [-(yi - w * xi) * xi for xi, yi in zip(Xb, yb)]   # per-example gradients
    clipped = np.array([clip(g, C) for g in grads])           # bound each one's influence
    noisy = clipped.sum() + rng.normal(0, sigma * C)          # add Gaussian noise to the sum
    return w - lr * noisy / len(Xb)

def train(sigma, C=1.0, lr=0.05, epochs=60):
    w = 0.0
    for _ in range(epochs):
        idx = rng.choice(len(Xd), 32, replace=False)
        w = dp_sgd_step(w, Xd[idx], yd[idx], lr, C, sigma)
    return w

print(f'no privacy   (sigma=0):   w = {train(0.0):.3f}  (true = {w_true})')
print(f'some privacy (sigma=1):   w = {train(1.0):.3f}')
print(f'more privacy (sigma=4):   w = {train(4.0):.3f}  (noisier -> less accurate)')

## 2 — The privacy–utility trade-off

More noise (≈ stronger privacy, smaller ε) means a worse model. We sweep the noise multiplier and
measure the error in the learned weight.

In [ ]:
for sigma in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:
    errs = [abs(train(sigma) - w_true) for _ in range(20)]
    print(f'noise sigma={sigma:4.1f}  ->  |w_error| = {np.mean(errs):.3f}  (more privacy = more error)')

### Validate: more privacy noise means a noisier estimate (privacy–utility trade-off)

DP-SGD adds two effects: **clipping** bounds each example's influence (a small bias), and
the **noise** $\sigma$ injects variance for privacy. More noise = stronger privacy = a
*noisier* estimate. We confirm the clip caps the gradient at $C$, then that the run-to-run
standard deviation of the learned weight grows with $\sigma$.

In [ ]:
# clipping bounds each per-example gradient's magnitude to C
assert abs(clip(5.0, 1.0)) <= 1.0 + 1e-9 and np.isclose(clip(0.3, 1.0), 0.3), 'clip caps at C'
# more noise -> higher variance of the learned weight across runs (the cost of privacy)
for sigma in [0.0, 2.0, 8.0, 20.0]:
    spread = np.std([train(sigma) for _ in range(200)])
    print(f'noise sigma={sigma:4.1f} -> std(w over runs) = {spread:.4f}')
low = np.std([train(0.0) for _ in range(200)])
high = np.std([train(20.0) for _ in range(200)])
assert high > low, 'more privacy noise makes the estimate noisier (less reliable)'
print('\n✅ the privacy–utility trade-off: more noise = more privacy = a noisier estimate')

## 3 — Federated Averaging (FedAvg)

Each client trains on its OWN data and sends back only weights; the server averages them (weighted
by data count). The raw data never leaves the client. We confirm the federated model approaches the
centralized one.

In [ ]:
# split the data across 5 clients (their data stays local)
clients = np.array_split(np.arange(len(Xd)), 5)

def local_train(w, idx, lr=0.05, steps=20):
    for _ in range(steps):
        g = -np.mean((yd[idx] - w * Xd[idx]) * Xd[idx])
        w -= lr * g
    return w

def fedavg(rounds=15):
    w = 0.0
    for _ in range(rounds):
        updates = [local_train(w, c) for c in clients]            # train locally
        counts = np.array([len(c) for c in clients])
        w = np.average(updates, weights=counts)                  # aggregate weights only
    return w

centralized = local_train(0.0, np.arange(len(Xd)), steps=300)
print(f'federated (FedAvg) weight:  {fedavg():.3f}')
print(f'centralized weight:         {centralized:.3f}')
print('FedAvg reaches the centralized solution without any client sharing its raw data.')

### Validate: FedAvg recovers the model without centralising data

Federated averaging trains on each client's *local* data and averages the weights (by data
count) — the raw data never moves. We confirm FedAvg reaches the true weight, matching what
centralised training would find, purely from local updates.

In [ ]:
w_fed = fedavg(rounds=15)
print(f'FedAvg estimate: {w_fed:.3f}  (true w = {w_true})')
assert abs(w_fed - w_true) < 0.1, 'FedAvg should recover the model from local updates alone'
# count-weighting: a client with more data pulls the average toward its local fit
big = np.arange(0, 150); small = np.arange(150, 200)
w_big = local_train(0.0, big); w_small = local_train(0.0, small)
w_weighted = (len(big) * w_big + len(small) * w_small) / (len(big) + len(small))
print(f'count-weighted avg {w_weighted:.3f} is pulled toward the larger client\'s fit {w_big:.3f}')
assert abs(w_weighted - w_big) < abs(w_weighted - w_small), 'the larger client dominates the average'
print('\n✅ FedAvg recovers the model from on-device updates, weighted by data count')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **privacy budget** | more noise = more privacy but less accuracy (verified); track the ε budget |
| **non-IID clients** | naive FedAvg fits no client well (demo); use FedProx/personalization |
| **communication cost** | FL sends model updates every round — compress/quantize them |
| **gradient clipping bias** | clipping distorts the gradient direction, not just magnitude |
| **updates still leak** | model updates can leak data; combine FL with DP for real privacy |

Demo: non-IID clients' local optima diverge, so their average fits neither.

In [ ]:
# The federated catch: FedAvg assumes clients have SIMILAR (IID) data. When clients are
# heterogeneous (non-IID) — each sees a different slice — naive averaging can pull toward a
# compromise that fits no client well. We show the averaged model between two skewed clients.
c_low  = np.arange(len(Xd))[Xd < 0]     # a client seeing only negative-x data
c_high = np.arange(len(Xd))[Xd >= 0]    # a client seeing only positive-x data
w_low, w_high = local_train(0.0, c_low), local_train(0.0, c_high)
print(f'client A (x<0) local w = {w_low:.3f}')
print(f'client B (x>=0) local w = {w_high:.3f}')
print(f'their average          = {0.5*(w_low+w_high):.3f}  (true {w_true})')
print('\nWhen clients are non-IID their local optima diverge, and the average can fit neither.')
print('Real FL uses more rounds, proximal terms (FedProx), or personalization to cope.')

## ✏️ Your turn

**Exercise.** Implement `clip_gradient(g, C)` (scale g down so its magnitude is at most C, leave it
unchanged otherwise) and `fedavg_aggregate(weights, counts)` (the data-count-weighted average of the
clients' weights). These are the cores of DP-SGD and federated learning.

In [ ]:
def clip_gradient(g, C):
    # TODO(you): if |g| > C scale it to norm C, else leave unchanged (works for scalar or vector g)
    return ...

def fedavg_aggregate(weights, counts):
    # TODO(you): weighted average of client weights, weighted by their data counts
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(clip_gradient(5.0, 1.0), 1.0)            # large gradient clipped to C
assert np.isclose(clip_gradient(0.3, 1.0), 0.3)           # small gradient untouched
assert np.isclose(np.linalg.norm(clip_gradient(np.array([3.0, 4.0]), 1.0)), 1.0)  # vector -> norm C
# FedAvg weights by data count: a client with more data pulls the average toward it
assert np.isclose(fedavg_aggregate([1.0, 3.0], [10, 30]), (1.0*10 + 3.0*30)/40)
print('\u2713 gradient clipping and FedAvg aggregation are correct')

<details>
<summary>Solution</summary>

```python
def clip_gradient(g, C):
    norm = np.linalg.norm(g)
    return g * min(1.0, C / (norm + 1e-12))

def fedavg_aggregate(weights, counts):
    return np.average(weights, weights=counts, axis=0)
```

Clipping bounds each example's influence (the sensitivity DP noise is calibrated to); FedAvg keeps
raw data on-device and shares only weights. Combined with secure aggregation and DP noise, they form
a layered privacy defense.

</details>

## Key takeaways

- **DP-SGD trades utility for privacy:** clip per-example gradients, add noise; more noise
  = stronger privacy = more error (verified) — there's no free privacy.
- **Federated learning keeps data on-device:** clients train locally, the server averages
  weights by data count (verified) — raw data never moves.
- **FedAvg assumes IID clients:** heterogeneous (non-IID) clients diverge and the naive
  average can fit none (demo) — real FL needs more rounds / FedProx / personalization.
- **Privacy and utility are in tension** — pick the operating point deliberately.